# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIRˆ² dataset using the `mlcroissant` library and Python. The dataset contains protein abundance and modification data derived from mass spectrometry analysis, annotated with a Croissant schema.

### Dataset Source
The dataset's Croissant schema is made available at:
`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and recordsets from the FAIRˆ² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
# Load the Croissant dataset
dataset = mlc.Dataset(url)

# Print dataset title and description
md = dataset.metadata
print(f"Dataset: {md.name}\nDescription: {md.description}")

## 2. Data Overview
Explore available record sets in the dataset and display their IDs. For each record set, print its fields and associated `@id`s.

In [ ]:
# List all record sets, fields, and columns by their @id
record_sets = list(dataset.metadata.record_sets)
print(f"Total record sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set: {rs['@id']} ({rs.get('name', 'Unnamed')})")
    if 'fields' in rs:
        for f in rs['fields']:
            # Each field reference is like {'@id': '...'}
            print(f"    Field: {f['@id']}")
    print()

## 3. Data Extraction
Load the data from a record set into a DataFrame for analysis. Use the record set and field `@id`s as referenced above. For demonstration, we'll use the main protein abundance record set.


In [ ]:
# Pick the main record set that contains protein measurements
# (You may need to adapt this ID to match the overview output -- here we use the first as a typical main table)
main_record_set_id = record_sets[0]['@id']

# Get all record set IDs for further exploration
record_set_ids = [rs['@id'] for rs in record_sets]

print('Available record sets:', record_set_ids)
dataframes = {}

# Load all record sets into pandas DataFrames
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"Loaded {len(dataframes[rs_id])} records from {rs_id}")

# Show columns for the main table
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply basic analysis: filtering on a numeric protein abundance column, normalization, and grouping. Refer to the columns by their field `@id` as listed above.

In [ ]:
# Example: filter, normalize and group by protein ID field
df = dataframes[main_record_set_id]

# Let us automatically pick a numeric field for exploration (e.g. 'abundance' or 'coverage_percentage')
# Show available columns
print("Available columns for EDA:", df.columns.tolist())

# We'll use the first float or integer column we find
numeric_field = None
for c in df.columns:
    if pd.api.types.is_numeric_dtype(df[c]):
        numeric_field = c
        break
if numeric_field is None:
    # If no numeric columns found, raise error to user
    raise ValueError('No numeric field found for analysis! Please check field types.')
print(f"Using numeric field '{numeric_field}' for filtering and normalization.")

# Filter records above a threshold
threshold = df[numeric_field].mean() if df[numeric_field].dtype != 'bool' else 0
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (mean): {len(filtered_df)} records")

# Normalize the numeric field (z-score)
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a categorical field, e.g. any column with "accession" or "id" in its name
group_field = None
for col in df.columns:
    if (col != numeric_field) and (('accession' in col.lower()) or ('id' in col.lower())):
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Grouped mean {numeric_field} by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions (e.g. histogram or boxplot) of a numeric abundance field and explore correlation if there is more than one numeric variable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the selected numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Frequency")
plt.show()

# If there is another numeric column, plot a scatter correlation
other_numeric = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != numeric_field]
if other_numeric:
    other_field = other_numeric[0]
    plt.figure(figsize=(6,4))
    sns.scatterplot(data=df, x=numeric_field, y=other_field)
    plt.title(f"{numeric_field} vs {other_field}")
    plt.show()

## 6. Conclusion
With `mlcroissant`, we've loaded, explored, and visualized the FAIRˆ² protein abundance dataset. This workflow can be adapted to analyze other structured datasets described using Croissant schemas.

- We inspected available record sets and fields via their `@id`s, guaranteeing clarity and reproducibility.
- We demonstrated typical EDA: numeric filtering, normalization, and aggregation/grouping.
- Basic visualizations (distribution and scatter) provided initial insights.

### Next Steps
- Dive deeper into domain-specific biological analysis using individual fields.
- Integrate this data into a larger pipeline for protein biomarker discovery or comparative studies.